In [1]:
# try:
#     %pip install --user "oracledb" --no-warn-script-location
# except Exception as e:
#     print("\x1b[31m\u2717 Unexpected error! Please contact course staff\n" +
#          "Please include the entire text above and below in your message.")
#     raise

# Data Insertion

In [2]:
import oracledb
import csv

In [133]:
dsn = oracledb.makedsn("localhost", 1522, service_name="stu")
connection = oracledb.connect(user="ora_lding04", password="a88101027", dsn=dsn)

In [134]:
cur = connection.cursor()

## IMDB TITLE BASICS

In [111]:
cur.execute("DROP TABLE imdb_title_basics")

DatabaseError: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/

In [114]:
cur.execute("""
    CREATE TABLE imdb_title_basics (
        tconst VARCHAR2(20) NOT NULL PRIMARY KEY,
        titleType VARCHAR2(50),
        primaryTitle VARCHAR2(500),
        originalTitle VARCHAR2(500),
        isAdult NUMBER(1),
        startYear NUMBER(4),
        endYear NUMBER(4),
        runtimeMinutes INTEGER,
        Genres VARCHAR2(500)
    )
""")

In [11]:
import pandas as pd
import numpy as np

In [82]:
# file_path = "../datasets/title.basics.tsv"
title_basic_df = pd.read_csv("Cleaned/imdb_title_basics_cleaned.csv")
title_basic_df = title_basic_df.replace({np.nan: None})
# title_basic_df_try = title_basic_df.iloc[[0]]
# with open(file_path, 'r', encoding='utf-8') as f:
#     reader = csv.reader(f, delimiter='\t') 
#     next(reader)
#     data = [[None if col == r'\N' else col for col in row] for row in reader]
#     cur.executemany("INSERT INTO imdb_title_basics VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)", data)

In [115]:
data_to_insert = title_basic_df.values.tolist()

In [116]:
cur.executemany("INSERT INTO imdb_title_basics VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)", data_to_insert)
connection.commit()

In [ ]:
# cur.execute("ALTER TABLE imdb_title_basics ADD CONSTRAINT uq_imdb_title UNIQUE (primaryTitle)")

DatabaseError: ORA-02299: cannot validate (ORA_LDING04.UQ_IMDB_TITLE) - duplicate keys found
Help: https://docs.oracle.com/error-help/db/ora-02299/

In [118]:
cur.execute("""
DELETE FROM imdb_title_basics
WHERE primaryTitle NOT IN (
    SELECT title FROM all_streaming_movies
)
""")
connection.commit()

## Rotten Tomatoes

In [122]:
rotten_tomatoes_df = pd.read_csv("Cleaned/rotten_tomatoes_movies_cleaned.csv")
rotten_tomatoes_df = rotten_tomatoes_df.replace({np.nan: None})
rotten_tomatoes_df = rotten_tomatoes_df.drop_duplicates(subset=['id'])

In [123]:
pd.DataFrame(rotten_tomatoes_df)

,id,title,audienceScore,tomatoMeter,maturity_rating,ratingContents,releaseDateTheaters,releaseDateStreaming,runtimeMinutes,genre,originalLanguage,director,writer,boxOffice,distributor,soundMix
0,adrift_2018,Adrift,65.0,69.0,PG-13,"['Injury Images', 'Brief Drug Use', 'Thematic ...",2018-06-01,2018-08-21,120.0,"Adventure, Drama, Romance",English,Baltasar Kormákur,"Aaron Kandell,Jordan Kandell,David Branson Smith",$31.4M,STX Films,None
1,robin_hood_2018,Robin Hood,40.0,15.0,PG-13,"['Extended Sequences of Violence', 'Action', '...",2018-11-21,2019-02-19,118.0,"Action, Adventure",English,Otto Bathurst,"Ben Chandler,David James Kelly",$30.8M,Lionsgate Films,Dolby Atmos
2,think_like_a_dog,Think Like a Dog,52.0,70.0,PG,['Rude and Suggestive Material'],None,2020-06-08,91.0,"Kids & family, Comedy, Fantasy",English,Gil Junger,"Gil Junger,John J. Strauss",None,None,None
3,hes_out_there,He's Out There,32.0,43.0,R,"['Horror Violence', 'Terror', 'Some Language']",2018-09-14,2018-10-16,89.0,"Horror, Mystery & thriller",English,Quinn Lasher,Mike Scannell,None,Vertical Entertainment,None
4,skate_kitchen,Skate Kitchen,76.0,89.0,R,"['Language Throughout', 'Drug Use', 'Some Nudi...",2018-08-10,2018-11-20,106.0,Drama,English,Crystal Moselle,"Crystal Moselle,Aslihan Unaldi,Jen Silverman",$236.1K,Magnolia Pictures,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1637,godmothered,Godmothered,57.0,67.0,PG,['Some Suggestive Comments'],None,2020-12-04,110.0,"Comedy, Fantasy",English,Sharon Maguire,"Kari Granlund,Melissa Stack",None,None,"Dolby Digital, Dolby Atmos"
1638,destroyer,Destroyer,50.0,74.0,R,"['Language Throughout', 'Brief Drug Use', 'Som...",2018-12-25,2019-04-23,123.0,"Crime, Drama, Mystery & thriller, Action",English,Karyn Kusama,"Phil Hay,Matt Manfredi",$1.5M,Annapurna Pictures,None
1639,boundaries_2018,Boundaries,56.0,49.0,R,"['Language', 'Drug Material', 'Nude Sketches',...",2018-06-22,2018-10-16,104.0,"Comedy, Drama",English,Shana Feste,Shana Feste,$698.4K,Sony Pictures Classics,None
1640,211,211,10.0,4.0,R,"['Violence', 'Language Throughout']",2018-06-08,2018-06-08,83.0,"Action, Mystery & thriller",English,York Alec Shackleton,None,None,Momentum Pictures,None


In [103]:
cur.execute("DROP TABLE rotten_tomatoes")

DatabaseError: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/

In [124]:
cur.execute("""
    CREATE TABLE rotten_tomatoes (
        id VARCHAR2(200) NOT NULL PRIMARY KEY,
        Title VARCHAR2(500),
        audienceScore INTEGER,
        tomatoMeter INTEGER,
        maturity_rating VARCHAR2(10),
        ratingContents VARCHAR2(1000),
        releaseDateTheaters DATE,
        releaseDateStreaming DATE, 
        runtimeMinutes INTEGER,
        genre VARCHAR2(100),
        originalLanguage VARCHAR2(50),
        director VARCHAR2(200),
        writer VARCHAR2(200),
        boxOffice VARCHAR2(50),
        distributor VARCHAR2(200),
        soundMix VARCHAR2(200)
    )
""")

In [125]:
data_to_insert_rt = rotten_tomatoes_df.values.tolist()

In [127]:
cur.executemany("INSERT INTO rotten_tomatoes VALUES (:1, :2, :3, :4, :5, :6, TO_DATE(:7, 'YYYY-MM-DD'), TO_DATE(:8, 'YYYY-MM-DD'), :9, :10, :11, :12, :13, :14, :15, :16)", data_to_insert_rt)
connection.commit()

In [128]:
cur.execute("""
DELETE FROM rotten_tomatoes
WHERE Title NOT IN (
    SELECT title FROM all_streaming_movies
)
""")
connection.commit()

## DISNEY PLUS

In [57]:
cur.execute("DROP TABLE disney_plus")

DatabaseError: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/

In [58]:
cur.execute("""
    CREATE TABLE disney_plus (
        show_id VARCHAR2(20) NOT NULL PRIMARY KEY,
        Type VARCHAR2(50),
        Title VARCHAR2(500),
        Director VARCHAR2(500),
        Cast VARCHAR2(500),
        Country VARCHAR2(200),
        Date_added DATE,
        release_year NUMBER(4),
        matuarity_rating VARCHAR2(10),
        duration VARCHAR2(50),
        listed_in VARCHAR2(500),
        description VARCHAR(1000)
    )
""")

In [59]:
disney_plus_df = pd.read_csv("Cleaned/disney_cleaned.csv")
disney_plus_df = disney_plus_df.replace({np.nan: None})

In [60]:
pd.DataFrame(disney_plus_df)

,show_id,type,title,director,cast,country,date_added,release_year,maturity_rating,duration,listed_in,description
0,s1,Movie,Duck the Halls: A Mickey Mouse Christmas Special,"Alonso Ramirez Ramos, Dave Wasson","Chris Diamantopoulos, Tony Anselmo, Tress MacN...",None,2021-11-26,2016,TV-G,None,"Animation, Family",Join Mickey and the gang as they duck the halls!
1,s2,Movie,Ernest Saves Christmas,John Cherry,"Jim Varney, Noelle Parker, Douglas Seale",None,2021-11-26,1988,PG,None,Comedy,Santa Claus passes his magic bag to a new St. ...
2,s3,Movie,Ice Age: A Mammoth Christmas,Karen Disher,"Raymond Albert Romano, John Leguizamo, Denis L...",United States,2021-11-26,2011,TV-G,None,"Animation, Comedy, Family",Sid the Sloth is on Santa's naughty list.
3,s4,Movie,The Queen Family Singalong,Hamish Hamilton,"Darren Criss, Adam Lambert, Derek Hough, Alexa...",None,2021-11-26,2021,TV-PG,None,Musical,"This is real life, not just fantasy!"
4,s5,TV Show,The Beatles: Get Back,None,"John Lennon, Paul McCartney, George Harrison, ...",None,2021-11-25,2021,None,None,"Docuseries, Historical, Music",A three-part documentary from Peter Jackson ca...
...,...,...,...,...,...,...,...,...,...,...,...,...
1442,s1446,Movie,X-Men Origins: Wolverine,Gavin Hood,"Hugh Jackman, Liev Schreiber, Danny Huston, wi...","United States, United Kingdom",2021-06-04,2009,PG-13,None,"Action-Adventure, Family, Science Fiction",Wolverine unites with legendary X-Men to fight...
1443,s1447,Movie,Night at the Museum: Battle of the Smithsonian,Shawn Levy,"Ben Stiller, Amy Adams, Owen Wilson, Hank Azar...","United States, Canada",2021-04-02,2009,PG,None,"Action-Adventure, Comedy, Family",Larry Daley returns to rescue some old friends...
1444,s1448,Movie,Eddie the Eagle,Dexter Fletcher,"Tom Costello, Jo Hartley, Keith Allen, Dickon ...","United Kingdom, Germany, United States",2020-12-18,2016,PG-13,None,"Biographical, Comedy, Drama","True story of Eddie Edwards, a British ski-jum..."
1445,s1449,Movie,Bend It Like Beckham,Gurinder Chadha,"Parminder Nagra, Keira Knightley, Jonathan Rhy...","United Kingdom, Germany, United States",2020-09-18,2003,PG-13,None,"Buddy, Comedy, Coming of Age",Despite the wishes of their traditional famili...


In [61]:
data_to_insert_disney = disney_plus_df.values.tolist()

In [62]:
cur.executemany("INSERT INTO disney_plus VALUES (:1, :2, :3, :4, :5, :6, TO_DATE(:7, 'YYYY-MM-DD'), :8, :9, :10, :11, :12)", data_to_insert_disney)
connection.commit()

## NETFLIX_1

In [63]:
cur.execute("DROP TABLE netflix_1")

DatabaseError: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/

In [64]:
cur.execute("""
    CREATE TABLE netflix_1 (
        show_id VARCHAR2(20) NOT NULL PRIMARY KEY,
        Type VARCHAR2(50),
        Title VARCHAR2(500),
        Director VARCHAR2(500),
        Cast VARCHAR2(1000),
        Country VARCHAR2(200),
        Date_added DATE,
        release_year NUMBER(4),
        matuarity_rating VARCHAR2(10),
        duration VARCHAR2(50),
        genres VARCHAR2(500),
        description VARCHAR(1000)
    )
""")

In [65]:
netflix_1_df = pd.read_csv("Cleaned/netflix_1_cleaned.csv")
netflix_1_df = netflix_1_df.replace({np.nan: None})

In [66]:
pd.DataFrame(netflix_1_df)

,show_id,type,title,director,cast,country,date_added,release_year,maturity_rating,duration,genres,description
0,s1,TV Show,3%,None,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...
2,s1001,TV Show,Blue Planet II,None,David Attenborough,United Kingdom,2018-12-03,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...
3,s1002,Movie,Blue Ruin,Jeremy Saulnier,"Macon Blair, Devin Ratray, Amy Hargreaves, Kev...","United States, France",2019-02-25,2013,R,90,"Independent Movies, Thrillers",Bad news from the past unhinges vagabond Dwigh...
4,s1003,Movie,Blue Streak,Les Mayfield,"Martin Lawrence, Luke Wilson, Peter Greene, Da...","Germany, United States",2021-01-01,1999,PG-13,94,"Action & Adventure, Comedies",A jewel thief returns to his hiding place afte...
...,...,...,...,...,...,...,...,...,...,...,...,...
5917,s993,TV Show,Bloodride,None,"Ine Marie Wilmann, Bjørnar Teigen, Emma Spetal...",Norway,2020-03-13,2020,TV-MA,1,"International TV Shows, TV Horror, TV Mysteries",The doomed passengers aboard a spectral bus he...
5918,s994,Movie,Blow,Ted Demme,"Johnny Depp, Penélope Cruz, Franka Potente, Ra...",United States,2019-10-01,2001,R,123,Dramas,Cocaine smuggler George rises from poverty to ...
5919,s995,TV Show,Blown Away,None,None,Canada,2019-07-12,2019,TV-14,1,"International TV Shows, Reality TV",Ten master artists turn up the heat in glassbl...
5920,s996,TV Show,Blue Exorcist,None,"Nobuhiko Okamoto, Jun Fukuyama, Kana Hanazawa,...",Japan,2020-09-01,2017,TV-MA,2,"Anime Series, International TV Shows",Determined to throw off the curse of being Sat...


In [67]:
data_to_insert_netflix = netflix_1_df.values.tolist()

In [68]:
cur.executemany("INSERT INTO netflix_1 VALUES (:1, :2, :3, :4, :5, :6, TO_DATE(:7, 'YYYY-MM-DD'), :8, :9, :10, :11, :12)", data_to_insert_netflix)
connection.commit()

## NETFLIX_2

In [69]:
netflix_2_df = pd.read_csv("Cleaned/netflix_2_cleaned.csv")
netflix_2_df = netflix_2_df.replace({np.nan: None})

In [70]:
pd.DataFrame(netflix_2_df)

,show_id,type,title,director,cast,country,date_added,release_year,maturity_rating,duration,listed_in,description
0,81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China",2019-09-09,2019,TV-PG,None,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
1,70234439,TV Show,Transformers Prime,None,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,2018-09-08,2013,TV-Y7-FV,None,Kids' TV,"With the help of three human allies, the Autob..."
2,80058654,TV Show,Transformers: Robots in Disguise,None,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,2018-09-08,2016,TV-Y7,None,Kids' TV,When a prison ship crash unleashes hundreds of...
3,80244601,TV Show,Castle of Stars,None,"Chaiyapol Pupart, Jintanutda Lummakanon, Worra...",None,2018-09-07,2015,TV-14,None,"International TV Shows, Romantic TV Shows, TV ...",As four couples with different lifestyles go t...
4,80203094,Movie,City of Joy,Madeleine Gavin,None,"United States,",2018-09-07,2018,TV-MA,None,Documentaries,Women who've been sexually brutalized in war-t...
...,...,...,...,...,...,...,...,...,...,...,...,...
3855,80049281,Movie,Secret in Their Eyes,Billy Ray,"Chiwetel Ejiofor, Nicole Kidman, Julia Roberts...","United States, United Kingdom, Spain, South Korea",2018-04-01,2015,PG-13,None,"Dramas, Thrillers",A former FBI investigator reopens the haunting...
3856,70084328,Movie,The Aerial,Esteban Sapir,"Rafael Ferro, Sol Moreno, Jonathan Sandor, Ale...",Argentina,2018-04-01,2007,TV-MA,None,"Dramas, International Movies, Sci-Fi & Fantasy","In the City Without a Voice, only faceless sin..."
3857,70099610,Movie,The Duchess,Saul Dibb,"Keira Knightley, Ralph Fiennes, Charlotte Ramp...","United Kingdom, Italy, France, United States",2018-04-01,2008,PG-13,None,"Dramas, International Movies, Romantic Movies","To compensate for her unhappy marriage, young ..."
3858,80217041,Movie,The Search for Life in Space,Stephen Amezdroz,None,United States,2018-04-01,2016,TV-G,None,Documentaries,To determine whether we're alone in the univer...


In [71]:
cur.execute("DROP TABLE netflix_2")

DatabaseError: ORA-00942: table or view does not exist
Help: https://docs.oracle.com/error-help/db/ora-00942/

In [72]:
cur.execute("""
    CREATE TABLE netflix_2 (
        show_id VARCHAR2(20) NOT NULL PRIMARY KEY,
        Type VARCHAR2(50),
        Title VARCHAR2(500),
        Director VARCHAR2(500),
        Cast VARCHAR2(1000),
        Country VARCHAR2(200),
        Date_added DATE,
        release_year NUMBER(4),
        matuarity_rating VARCHAR2(10),
        duration VARCHAR2(50),
        listed_in VARCHAR2(500),
        description VARCHAR(1000)
    )
""")

In [73]:
data_to_insert_netflix_2 = netflix_2_df.values.tolist()

In [138]:
cur.executemany("INSERT INTO netflix_2 VALUES (:1, :2, :3, :4, :5, :6, TO_DATE(:7, 'YYYY-MM-DD'), :8, :9, :10, :11, :12)", data_to_insert_netflix_2)
connection.commit()

DatabaseError: ORA-01536: space quota exceeded for tablespace 'USERS'
Help: https://docs.oracle.com/error-help/db/ora-01536/

## Remove duplicate

In [80]:
cur.execute("""
    CREATE VIEW all_streaming_movies AS
SELECT show_id, title, director, country, release_year, 'Netflix' as source FROM netflix_1
UNION
SELECT show_id, title, director, country, release_year, 'Netflix' as source FROM netflix_2
UNION
SELECT show_id, title, director, country, release_year, 'Disney' as source FROM disney_plus
""")

In [99]:
cur.execute("""
            DELETE FROM netflix_2 
WHERE title IN (
    SELECT title FROM all_streaming_movies
)""")
connection.commit()

In [100]:
cur.execute("""
            DELETE FROM disney_plus 
WHERE title IN (SELECT title FROM netflix_1 UNION SELECT title FROM netflix_2)
""")
connection.commit()

In [120]:
# 1. Enable movement for all three tables
cur.execute("ALTER TABLE netflix_1 ENABLE ROW MOVEMENT")
cur.execute("ALTER TABLE netflix_2 ENABLE ROW MOVEMENT")
cur.execute("ALTER TABLE disney_plus ENABLE ROW MOVEMENT")
cur.execute("ALTER TABLE imdb_title_basics ENABLE ROW MOVEMENT")

# 2. Now run the shrink again
cur.execute("ALTER TABLE netflix_1 SHRINK SPACE")
cur.execute("ALTER TABLE netflix_2 SHRINK SPACE")
cur.execute("ALTER TABLE disney_plus SHRINK SPACE")
cur.execute("ALTER TABLE imdb_title_basics SHRINK SPACE")
# 3. Finalize and clear the trash
cur.execute("PURGE RECYCLEBIN")
connection.commit()

In [140]:
cur.execute("PURGE RECYCLEBIN")
connection.commit()

In [141]:
# Run this to actually see the MB_Used vs MB_Limit
space_df = pd.read_sql("SELECT bytes/1024/1024 as MB_Used, max_bytes/1024/1024 as MB_Limit FROM user_ts_quotas", connection)
print(space_df)

   MB_USED  MB_LIMIT
0        8         8


/var/folders/tr/w9p9c_hx39j1vrtknn93nc0c0000gn/T/ipykernel_50504/2360948130.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  space_df = pd.read_sql("SELECT bytes/1024/1024 as MB_Used, max_bytes/1024/1024 as MB_Limit FROM user_ts_quotas", connection)


In [130]:
imdb_ratings_df = pd.read_csv("Cleaned/imdb_title_ratings_cleaned.csv")
imdb_ratings_df = imdb_ratings_df.replace({np.nan: None})

In [131]:
pd.DataFrame(imdb_ratings_df)

,tconst,averageRating,numVotes
0,tt0069049,6.7,8464
1,tt0365545,6.4,10315
2,tt0800325,7.0,60014
3,tt0870154,6.5,225717
4,tt0914376,7.3,6952
...,...,...,...
5781,tt9906218,7.7,26
5782,tt9906260,9.8,157637
5783,tt9908860,7.2,4301
5784,tt9914156,6.4,163


In [135]:
cur.execute("""
    CREATE TABLE imdb_ratings (
        tconst VARCHAR2(20) NOT NULL PRIMARY KEY,
        average_ratings DECIMAL(3,1),
        numVotes INTEGER
    )
""")

In [136]:
data_to_insert_imdb_ratings = imdb_ratings_df.values.tolist()

In [137]:
cur.executemany("INSERT INTO imdb_ratings VALUES (:1, :2, :3)", data_to_insert_imdb_ratings)
connection.commit()

# Research Questions

## Question 1
What is the impact of the COVID-19 pandemic on changing audience/viewer sentiment towards all movies measured by average IMDb rating? 
## Hypothesis 
There is a statistically significant increase in the average IMDb rating for movies and TV shows released between the height of COVID-19 (2020, 2021) and 2018-2019, likely due to increased consumption and greater access to entertainment, with algorithms curated to users' tastes. \
Let the average imdb score in year (2020, 2021) to be $\mu$ and the average imdb score in year (2018, 2019) to be $\mu_0$.\
$H_0: \mu = \mu_0, H_A : \mu > \mu_0$.


In [144]:
cur.execute("""
            CREATE VIEW imdb_movies_ratings AS
            SELECT 
            b.tconst, 
            b.primaryTitle, 
            b.startYear, 
            b.genres,
            r.average_ratings, 
            r.numVotes
            FROM imdb_title_basics b
            INNER JOIN imdb_ratings r ON b.tconst = r.tconst
            """)

In [157]:
avg_rating_df = pd.read_sql("""
    SELECT average_ratings
    FROM imdb_movies_ratings
    WHERE startYear IN (2018, 2019)
""", connection)

print(avg_rating_df)

/var/folders/tr/w9p9c_hx39j1vrtknn93nc0c0000gn/T/ipykernel_50504/1955979161.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  avg_rating_df = pd.read_sql("""


      AVERAGE_RATINGS
0                 6.7
1                 6.4
2                 7.0
3                 7.3
4                 7.6
...               ...
3408              7.7
3409              9.8
3410              7.2
3411              6.4
3412              6.1

[3413 rows x 1 columns]


In [158]:
avg_rating_df_muA = pd.read_sql("""
    SELECT average_ratings
    FROM imdb_movies_ratings
    WHERE startYear IN (2020, 2021)
""", connection)
print(avg_rating_df_muA)

/var/folders/tr/w9p9c_hx39j1vrtknn93nc0c0000gn/T/ipykernel_50504/1554226271.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  avg_rating_df_muA = pd.read_sql("""


      AVERAGE_RATINGS
0                 6.5
1                 7.5
2                 6.2
3                 4.7
4                 7.4
...               ...
2343              6.2
2344              6.5
2345              3.8
2346              7.8
2347              5.2

[2348 rows x 1 columns]


We then perform hypothesis testing by an independent two-sample t-test. Firstly, we set the significance level $\alpha = 5\%$ 

In [149]:
from scipy import stats

In [159]:
t_stat, p_value = stats.ttest_ind(avg_rating_df, avg_rating_df_muA, equal_var=False)

In [160]:
print(t_stat)

[-2.49862721]


In [161]:
print(p_value)

[0.01250003]


Since our p value is less than $\alpha$, we have enough evidence to reject our null hypothesis, and there is a statistically significant difference between the average IMDb ratings of movies released in 2018–2019 versus those released in 2020–2021.